# AIRL2 applied workflow

This notebook fits the canonical AIRL2 wrapper on a small anchored latent-segment dynamic choice problem. It demonstrates the public import, segment outputs, and a changed-reward policy calculation.

In [1]:
from pathlib import Path
import jax.numpy as jnp
import numpy as np
import econirl
from econirl import AIRL2, RewardSpec
from econirl.core.bellman import SoftBellmanOperator
from econirl.core.solvers import value_iteration
from econirl.environments.content_consumption import content_consumption
from econirl.simulation.synthetic import simulate_mixture_panel

package_path = Path(econirl.__file__).resolve()
repo_src = (Path.cwd() / "src").resolve()
print("Installed package import:", repo_src not in package_path.parents)
print("Canonical estimator:", AIRL2.__name__)

Installed package import: True
Canonical estimator: AIRL2


In [2]:
binge = content_consumption(theta=np.array([4.0, 0.05, 0.1, 0.2]), seed=0)
sampler = content_consumption(theta=np.array([0.5, 3.0, 3.0, 0.3]), seed=0)
panel = simulate_mixture_panel(
    [binge, sampler],
    [0.5, 0.5],
    n_individuals=250,
    n_periods=40,
    seed=42,
)
n_states = binge.num_states
n_actions = binge.num_actions
transitions = np.asarray(binge.transition_matrices)
reward_spec = RewardSpec.state_action_dependent(
    jnp.asarray(binge.feature_matrix),
    list(binge.parameter_names),
)
print("Panel:", panel.num_individuals, "individuals,", panel.num_observations, "transitions")
print("Problem shape:", n_states, "states by", n_actions, "actions")

Panel: 250 individuals, 10000 transitions
Problem shape: 65 states by 4 actions


In [3]:
model = AIRL2(
    n_states=n_states,
    n_actions=n_actions,
    discount=float(binge.problem_spec.discount_factor),
    num_segments=2,
    exit_action=binge.leave_action,
    absorbing_state=binge.session_ended_state,
    max_em_iterations=20,
    max_airl_rounds=20,
    initialization="random",
    compute_se=False,
    seed=7,
    verbose=False,
).fit(panel, transitions=transitions, reward=reward_spec)
print("Converged:", model.converged_)
print("EM iterations:", model.n_iter_)
print("Segment priors:", np.round(model.segment_priors_, 3))
print("Segment policy shape:", model.segment_policies_.shape)

Converged: True
EM iterations: 4
Segment priors: [0.448 0.552]
Segment policy shape: (2, 65, 4)


In [4]:
operator = SoftBellmanOperator(binge.problem_spec, jnp.asarray(transitions))
baseline = model.segment_policies_[0]
changed_reward = jnp.asarray(model.segment_reward_matrices_[0]).at[:, 0].add(0.25)
changed_reward = changed_reward.at[:, binge.leave_action].set(0.0)
changed = value_iteration(operator, changed_reward).policy
policy_shift = 0.5 * np.mean(np.abs(np.asarray(changed) - baseline).sum(axis=1))
print("Segment 0 reward anchor:", float(np.max(np.abs(model.segment_reward_matrices_[0, :, binge.leave_action]))))
print("Changed-reward policy TV:", round(float(policy_shift), 4))

Segment 0 reward anchor: 0.0
Changed-reward policy TV: 0.0466


## Manager interpretation

AIRL2 returns separate policies and anchored reward matrices for latent behavioral segments. Segment labels are arbitrary, so downstream comparisons must align segments before reading type-specific effects. The policy shift above is a model-based scenario calculation, not a causal effect from an uncontrolled intervention.